In [8]:
from collections import namedtuple
from typing import NamedTuple
import datetime
from dateutil.parser import parse
from typing import List, Optional

In [9]:
class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

    def is_high_tech(self) -> bool:
        return self.symbol in ['MSFT', 'GOOG', 'FB', 'AMZN', 'AAPL']

price = StockPrice(
    'MSFT',
    datetime.date(2018, 12, 14),
    106.03
)

assert price.symbol == 'MSFT'
assert price.closing_price == 106.03
assert price.is_high_tech()

In [ ]:
# Parse without checking, crashes

def parse_row(row: List[str]) -> StockPrice:
    symbol, date, closing_price = row
    return StockPrice(
        symbol=symbol, 
        date=parse(date).date(), 
        closing_price=float(closing_price)
    )

stock = parse_row(['MSFT', '2018-12-14', '106.03'])

assert stock.symbol == 'MSFT'
assert stock.date == datetime.date(2018, 12, 14)
assert stock.closing_price == 106.03

In [13]:
# Prefer rather getting None to crashing, validation

import re

def try_parse_row(row: List[str]) -> Optional[StockPrice]:
    symbol, date, closing_price = row

    # Stock synbol should be all CAPS
    if not re.match("^[A-Z]+$", symbol):
        return None

    try:
        date = parse(date).date()
    except ValueError:
        return None

    try:
        closing_price = float(closing_price)
    except ValueError:
        return None

    return StockPrice(symbol, date, closing_price)

# Should return Nine for errors
assert try_parse_row(["MSFT0", "2018-12-14", "106.03"]) is None
assert try_parse_row(["MSFT", "2018-12--14", "106.03"]) is None
assert try_parse_row(["MSFT", "2018-12-14", "x"]) is None

# But should return same as before if data is good.
assert try_parse_row(["MSFT", "2018-12-14", "106.03"]) == stock

In [15]:
import csv

clean_data: List[StockPrice] = []

with open("stocks.csv") as f:
    reader = csv.reader(f)
    for row in reader:
        maybe_stock = try_parse_row(row)
        if maybe_stock is None:
            print(f"Skipping invalid row: {row}")
        else:
            clean_data.append(maybe_stock)

Skipping invalid row: ['MSFT', '6/19/2014', 'n/a']


In [16]:
print(clean_data)

[StockPrice(symbol='AAPL', date=datetime.date(2014, 6, 20), closing_price=90.91), StockPrice(symbol='MSFT', date=datetime.date(2014, 6, 20), closing_price=41.68), StockPrice(symbol='FB', date=datetime.date(3014, 6, 20), closing_price=64.5), StockPrice(symbol='AAPL', date=datetime.date(2014, 6, 19), closing_price=91.86), StockPrice(symbol='FB', date=datetime.date(2014, 6, 19), closing_price=64.34)]
